# 01 - Data Download and Preprocessing

This notebook:
- Verifies that raw datasets are present in `data/raw/`.
- Preprocesses images (resize to 256×256, convert to RGB).
- Splits processed images into train and validation sets.


In [1]:
# imports and paths
import os
from pathlib import Path
from PIL import Image
import random

random.seed(42)

RAW_DIR = Path("../data/raw")
PROCESSED_DIR = Path("../data/processed")

TARGET_SIZE = (256, 256)
VAL_SPLIT = 0.1  # 10% validation


In [2]:
# Verify dataset folders exist
for folder in [
    RAW_DIR / "flickr",
    RAW_DIR / "wikiart",
    RAW_DIR / "danbooru"
]:
    if folder.exists():
        print(f"✓ Found: {folder}")
    else:
        print(f"✗ Missing: {folder}")

✓ Found: ..\data\raw\flickr
✓ Found: ..\data\raw\wikiart
✓ Found: ..\data\raw\danbooru


In [3]:
# preprocess a folder
def preprocess_folder(input_dir, output_dir,
                      target_size=TARGET_SIZE,
                      max_images=None):

    input_dir = Path(input_dir)
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    # Only collect image files
    image_extensions = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

    image_paths = [
        p for p in input_dir.iterdir()
        if p.is_file() and p.suffix.lower() in image_extensions
    ]

    print(f"Found {len(image_paths)} images in {input_dir}")

    # Randomly sample if requested
    if max_images is not None and len(image_paths) > max_images:
        image_paths = random.sample(image_paths, max_images)
        print(f"Using random sample of {len(image_paths)} images")

    # Process images
    for i, img_path in enumerate(image_paths, start=1):

        out_path = output_dir / img_path.name

        # Skip already processed images
        if out_path.exists():
            continue

        try:
            img = Image.open(img_path).convert("RGB")
            img = img.resize(target_size)
            img.save(out_path)

            if i % 500 == 0:
                print(f"Processed {i}/{len(image_paths)} images")

        except Exception as e:
            print(f"Skipping {img_path}: {e}")

    print("Done!\n")

In [4]:
# Verify dataset folders exist
for folder in [
    RAW_DIR / "flickr" / "flickr30k_images",
    RAW_DIR / "wikiart" / "wikiart-art-movementsstyles" / "Japanese_Art",
    RAW_DIR / "danbooru" / "portraits"
]:
    if folder.exists():
        print(f"✓ Found: {folder}")
    else:
        print(f"✗ Missing: {folder}")

✓ Found: ..\data\raw\flickr\flickr30k_images
✓ Found: ..\data\raw\wikiart\wikiart-art-movementsstyles\Japanese_Art
✓ Found: ..\data\raw\danbooru\portraits


In [5]:
import random
import shutil
from pathlib import Path

random.seed(42)

def create_cyclegan_dataset(source_a, source_b, output_root, test_split=0.1):
    """
    Creates a CycleGAN dataset structure:
        trainA, trainB, testA, testB

    source_a = Source domain (Photos)
    source_b = Target domain (Japanese Art or Anime)
    """

    source_a = Path(source_a)
    source_b = Path(source_b)
    output_root = Path(output_root)

    # Create folders
    for folder in ["trainA", "trainB", "testA", "testB"]:
        (output_root / folder).mkdir(parents=True, exist_ok=True)

    # Get image lists
    images_a = [p for p in source_a.iterdir() if p.is_file()]
    images_b = [p for p in source_b.iterdir() if p.is_file()]

    random.shuffle(images_a)
    random.shuffle(images_b)

    split_a = int(len(images_a) * (1 - test_split))
    split_b = int(len(images_b) * (1 - test_split))

    trainA = images_a[:split_a]
    testA = images_a[split_a:]

    trainB = images_b[:split_b]
    testB = images_b[split_b:]

    # Copy images
    for img in trainA:
        shutil.copy2(img, output_root / "trainA" / img.name)

    for img in testA:
        shutil.copy2(img, output_root / "testA" / img.name)

    for img in trainB:
        shutil.copy2(img, output_root / "trainB" / img.name)

    for img in testB:
        shutil.copy2(img, output_root / "testB" / img.name)

    print(f"\nCreated: {output_root.name}")
    print(f"trainA : {len(trainA)}")
    print(f"trainB : {len(trainB)}")
    print(f"testA  : {len(testA)}")
    print(f"testB  : {len(testB)}")

In [6]:
create_cyclegan_dataset(
    PROCESSED_DIR / "temp_photos",
    PROCESSED_DIR / "temp_japanese",
    PROCESSED_DIR / "japanese_style"
)


Created: japanese_style
trainA : 1800
trainB : 2011
testA  : 200
testB  : 224


In [7]:
create_cyclegan_dataset(
    PROCESSED_DIR / "temp_photos",
    PROCESSED_DIR / "temp_anime",
    PROCESSED_DIR / "anime_style"
)


Created: anime_style
trainA : 1800
trainB : 4500
testA  : 200
testB  : 500


In [8]:
from pathlib import Path

for dataset in ["japanese_style", "anime_style"]:
    print(f"\n{dataset}")

    root = PROCESSED_DIR / dataset

    for folder in ["trainA", "trainB", "testA", "testB"]:
        count = len(list((root / folder).glob("*")))
        print(f"{folder}: {count}")


japanese_style
trainA: 1800
trainB: 2011
testA: 200
testB: 224

anime_style
trainA: 1800
trainB: 4500
testA: 200
testB: 500


In [9]:
# Preprocess Flickr photos
preprocess_folder(
    RAW_DIR / "flickr" / "flickr30k_images" / "flickr30k_images",
    PROCESSED_DIR / "temp_photos",
    max_images=2000
)

preprocess_folder(
    RAW_DIR / "wikiart" /
    "wikiart-art-movementsstyles" /
    "Japanese_Art" /
    "Japanese_Art",
    PROCESSED_DIR / "temp_japanese"
)

preprocess_folder(
    RAW_DIR / "danbooru" / "portraits",
    PROCESSED_DIR / "temp_anime",
    max_images=5000
)

Found 31783 images in ..\data\raw\flickr\flickr30k_images\flickr30k_images
Using random sample of 2000 images
Processed 500/2000 images
Processed 1000/2000 images
Processed 1500/2000 images
Processed 2000/2000 images
Done!

Found 2235 images in ..\data\raw\wikiart\wikiart-art-movementsstyles\Japanese_Art\Japanese_Art
Processed 500/2235 images
Processed 1000/2235 images
Processed 1500/2235 images
Processed 2000/2235 images
Done!

Found 302652 images in ..\data\raw\danbooru\portraits
Using random sample of 5000 images
Processed 500/5000 images
Processed 1000/5000 images
Processed 1500/5000 images
Processed 2000/5000 images
Processed 2500/5000 images
Processed 3000/5000 images
Processed 3500/5000 images
Processed 4000/5000 images
Processed 4500/5000 images
Processed 5000/5000 images
Done!



In [10]:
import shutil

for folder in [
    PROCESSED_DIR / "temp_photos",
    PROCESSED_DIR / "temp_japanese",
    PROCESSED_DIR / "temp_anime",
]:
    if folder.exists():
        shutil.rmtree(folder)
        print(f"Deleted {folder.name}")

print("Temporary folders removed.")

Deleted temp_photos
Deleted temp_japanese
Deleted temp_anime
Temporary folders removed.
